In [ ]:
# FIGURE 6

In [ ]:
import numpy as np

import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
import matplotlib as mpl
label_size = 20
tick_size = 18
legend_font = 18
labelpad = 3
mpl.rcParams["font.size"] = label_size
mpl.rcParams["axes.labelpad"] = labelpad
mpl.rcParams["axes.labelsize"] = label_size
mpl.rcParams["legend.fontsize"] = legend_font
mpl.rcParams["xtick.labelsize"] = tick_size
mpl.rcParams["ytick.labelsize"] = tick_size

mpl.rcParams["boxplot.boxprops.linewidth"] = 1

box_width = 0.7

In [ ]:
import sys
sys.path.append("../python/")

from distance_stats import *

In [ ]:
def get_refinement_sizes(results_dir, match_types, m_vals=range(2,21)):
    refinement_sizes = dict()
    for m in range(2, 21):
        print(f"m: {m}")
        refinement_sizes[m] = {match_type: {"mean": 0.0, "std": 0.0} for match_type in match_types}
        matching_indices = {match_type: read_refined_sets(results_dir, f"n-{n}_m-{m}_k-{k}_non-uniform_exhaustive_projections_matching_{match_type}_indices.csv") for match_type in match_types if match_type not in ["all", "both"]}
        # TODO: GET NUMBER OF TWINS
        filename =  f"n-{n}_m-{m}_k-{k}_non-uniform_exhaustive_projections.csv"
        all_data = read_data(results_dir + filename, xgi_hypergraphs=False)
        refinement_sizes[m]["all"]["lengths"] = list(all_data["total_twins"])
        refinement_sizes[m]["all"]["mean"] = np.mean(all_data["total_twins"])
        refinement_sizes[m]["all"]["std"] = np.std(all_data["total_twins"])
        for match_type in match_types:
            if match_type in ["m", "diag"]:
                refinement_lengths = [len(minds) for minds in matching_indices[match_type]]
                refinement_sizes[m][match_type]["lengths"] = list(refinement_lengths)
                refinement_sizes[m][match_type]["mean"] = np.mean(refinement_lengths)
                refinement_sizes[m][match_type]["std"] = np.std(refinement_lengths)
            elif match_type == "both":
                m_indices = matching_indices["m"]
                diag_indices = matching_indices["diag"]
                assert(len(m_indices) == len(diag_indices))
                both_indices = []
                for proj_idx in range(len(m_indices)):
                    both_indices.append(sorted(list(set(m_indices[proj_idx]).intersection(diag_indices[proj_idx]))))
                refinement_lengths = [len(minds) for minds in both_indices]
                refinement_sizes[m][match_type]["lengths"] = list(refinement_lengths)
                refinement_sizes[m][match_type]["mean"] = np.mean(refinement_lengths)
                refinement_sizes[m][match_type]["std"] = np.std(refinement_lengths)

    return refinement_sizes

In [ ]:
exhaustive_dir = '../../results/reproduce/exhaustive/'
results_dir = exhaustive_dir
stats_dir      = '../../results/reproduce/distance-stats/'
dist_dir       = '../../results/reproduce/distance-matrices/'

In [ ]:
plot_dir = "../plots/"
k = 3
n = 6
distance_names = ["hyperNS", "hyperPD", "jaccard"]

match_types = ["all", "m", "diag", "both"]

stat_key = "mean"
max_m = 13
m_vals = range(2, max_m+1)

x_key = "m"
stat_key = "mean"

In [ ]:
refinement_sizes = get_refinement_sizes(results_dir, match_types)

In [ ]:
output_datas = read_output_datas(k, n, m_vals, results_dir)

In [ ]:
titles = {
    "all": "All Twins",
    "m": "Twins w/ same\n" +  r"$m$ as input",
    "diag": "Twins w/ same\n digaonal as input",
    "both": "Twins w/ same\n" +  "$m$ and diagonal"
}

In [ ]:
read_stats = True
write_stats = False

In [ ]:
fig = plt.figure(figsize=(16, 11), constrained_layout=True)
gs = fig.add_gridspec(4, len(distance_names)+2, width_ratios=[0.12, 0.22, 0.22, 0.22, 0.22], wspace=0.1, hspace=0.05)

row = 0

match_types = ["all", "m", "diag", "both"]
dist_labels = {
    "hyperNS": "NetSimile",
    "hyperPD": "Portrait\nDivergence",
    "jaccard": "Jaccard",
}

if x_key == "max_width":
    xlabel = "Worst Case Comlexity"
elif x_key == "num_size_dists":
    xlabel = "Num Size Dists"
elif x_key == "num_cliques":
    xlabel = "Number of Cliques"
elif x_key == "m":
    xlabel = "m"

for match_type in match_types:
    col = 0
    ax = fig.add_subplot(gs[row, col])
    ax.text(0.15, 0.5, titles[match_type], transform=ax.transAxes, va="center", ha="center")
    ax.axis("off")

    col = 1
    x = m_vals
    y = []
    for m in x:
        y.append(refinement_sizes[m][match_type]["lengths"])

    ax = fig.add_subplot(gs[row, col])
    ax.boxplot(y, positions=x, showfliers=False, whis=(0,100), widths=box_width)
    ax.set_yscale('log')
    ax.spines[['right', 'top']].set_visible(False)
    ax.set_xticks(x[::2], x[::2]);
    ax.set_xlabel("$m$")
    ax.set_ylabel("# Twins")
    ax.set_ylim(10**-0.5, 10**7)
    if row == 0:
        ax.set_title("Twins by\nRefinement Level")

    
    for col in range(2, len(distance_names)+2):
        dist = distance_names[col-2]
        ax = fig.add_subplot(gs[row, col])
        pickle_file = stats_dir + f"n-{n}_m-{min(m_vals)}-{max(m_vals)}_k-{k}_non-uniform_exhaustive_projections_{dist}_{match_type}_stats"
        if not read_stats:
            # dist_dir is separate because the distance matrices are hundreds of GB and
            # live in their own directory; it defaults to results_dir for a flat layout.
            stats = get_stat_dist(n, k, output_datas, dist, match_type, m_vals,
                                  results_dir, dist_dir=dist_dir)
            if write_stats:
                write_stats_file(pickle_file + ".csv.zst", stats)
        else:
            stats = read_stats_file(pickle_file + ".csv.zst")

        dist_label = dist_labels[dist]
    
        # For each m
        x = []
        y = []
        sidx = 0
        for m in m_vals:
            # Get the distance values for this m
            mdists = []
            while sidx < len(stats["m"]) and stats["m"][sidx] == m:
                if dist == "jaccard":
                    mdists.append(1.0 - stats[stat_key][sidx])
                else:
                    mdists.append(stats[stat_key][sidx])
    
                sidx += 1
            
            if len(mdists) > 0:
                ax.boxplot(mdists, positions = [m], showfliers=False, whis=(0,100), widths=box_width)
                
        if row == 0:
            ax.set_title(dist_label)

        ax.set_yticks([])
        if col == 2:
            ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
            ax.set_ylabel("Distance", labelpad=4.0)

        ax.set_xlim((min(m_vals)-1, max(m_vals)+1))
        ax.set_xticks([])
        
        if row == len(match_types)-1:
            ax.set_xticks(m_vals[::2], m_vals[::2])
            ax.set_xlabel(xlabel)
            
        ax.set_ylim(-0.1, 1.1)
        ax.spines[['right', 'top']].set_visible(False)
    row += 1
    
output_filename = plot_dir + f"non-uniform_exhaustive_m-2-{max_m}_pairwise_distances_{x_key}_combined_boxplots.pdf"
fig.savefig(output_filename)